# Logistic Regression với PyTorch - Tutorial
## Ví dụ với dataset Iris

### Logistic Regression là gì?
- Là thuật toán classification đơn giản nhất trong Neural Networks
- Chỉ có **1 layer** duy nhất: Linear layer
- Công thức: `y = W*x + b` (giống linear regression)
- Dùng **Softmax** để chuyển output thành probabilities

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
import matplotlib.pyplot as plt

print(f"PyTorch version: {torch.__version__}")

## Bước 1: Load và chuẩn bị dữ liệu

In [ ]:
# Load dữ liệu Iris
data = pd.read_csv('../data/IRIS.csv')
print(data.head())
print(f"\nShape: {data.shape}")

In [ ]:
# Tách X (features) và y (labels)
X = data.iloc[:, :-1].values  # 4 cột đầu
y = data.iloc[:, -1].values   # Cột species

# Encode labels: Iris-setosa→0, Iris-versicolor→1, Iris-virginica→2
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y)

# Chuẩn hóa features (mean=0, std=1)
scaler = StandardScaler()
X = scaler.fit_transform(X)

# Chia train/test (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")

In [ ]:
# Chuyển sang PyTorch tensors
X_train_tensor = torch.FloatTensor(X_train)
y_train_tensor = torch.LongTensor(y_train)  # LongTensor cho labels
X_test_tensor = torch.FloatTensor(X_test)
y_test_tensor = torch.LongTensor(y_test)

print(f"X_train shape: {X_train_tensor.shape}")
print(f"y_train shape: {y_train_tensor.shape}")

## Bước 2: Định nghĩa Model

```
Logistic Regression architecture:
Input (4) → Linear → Output (3)
              ↓
         W*x + b
```

In [ ]:
class LogisticRegression(nn.Module):
    def __init__(self, input_dim, output_dim):
        super(LogisticRegression, self).__init__()
        # Chỉ có 1 linear layer duy nhất!
        self.linear = nn.Linear(input_dim, output_dim)
    
    def forward(self, x):
        # Forward pass: trả về logits (chưa qua softmax)
        return self.linear(x)

# Khởi tạo model
model = LogisticRegression(input_dim=4, output_dim=3)
print(model)
print(f"\nParameters: {sum(p.numel() for p in model.parameters())}")

## Bước 3: Loss Function và Optimizer

In [ ]:
# CrossEntropyLoss = Softmax + NLLLoss
# Nó tự động tính softmax nên output của model chỉ cần là logits
criterion = nn.CrossEntropyLoss()

# Optimizer: SGD hoặc Adam
optimizer = optim.SGD(model.parameters(), lr=0.01)
# optimizer = optim.Adam(model.parameters(), lr=0.01)  # Thường tốt hơn

print(f"Loss: {criterion}")
print(f"Optimizer: {optimizer}")

## Bước 4: Training Loop

**Quy trình training:**
1. **Forward pass**: Tính output từ input
2. **Tính loss**: So sánh output với label thực tế
3. **Backward pass**: Tính gradients
4. **Update weights**: Optimizer cập nhật parameters

In [ ]:
num_epochs = 1000
losses = []
accuracies = []

for epoch in range(num_epochs):
    # 1. Forward pass
    outputs = model(X_train_tensor)  # Shape: (120, 3)
    loss = criterion(outputs, y_train_tensor)
    
    # 2. Backward pass
    optimizer.zero_grad()  # Reset gradients về 0
    loss.backward()        # Tính gradients (dL/dW)
    optimizer.step()       # Cập nhật: W = W - lr * gradient
    
    # 3. Tính accuracy
    with torch.no_grad():
        _, predicted = torch.max(outputs, 1)  # Lấy class có score cao nhất
        accuracy = (predicted == y_train_tensor).sum().item() / len(y_train_tensor) * 100
    
    losses.append(loss.item())
    accuracies.append(accuracy)
    
    if (epoch + 1) % 100 == 0:
        print(f'Epoch {epoch+1:4d} | Loss: {loss.item():.4f} | Acc: {accuracy:.2f}%')

print("\n✅ Training hoàn thành!")

## Bước 5: Visualize

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(losses, color='red', linewidth=2)
ax1.set_title('Training Loss', fontsize=14, fontweight='bold')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.grid(True, alpha=0.3)

ax2.plot(accuracies, color='blue', linewidth=2)
ax2.set_title('Training Accuracy', fontsize=14, fontweight='bold')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy (%)')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Bước 6: Evaluation

In [ ]:
model.eval()  # Chuyển sang evaluation mode

with torch.no_grad():
    test_outputs = model(X_test_tensor)
    _, predicted = torch.max(test_outputs, 1)
    
    test_accuracy = (predicted == y_test_tensor).sum().item() / len(y_test_tensor) * 100
    
    print(f"Test Accuracy: {test_accuracy:.2f}%")
    print(f"\nDự đoán: {predicted.numpy()}")
    print(f"Thực tế:  {y_test_tensor.numpy()}")

## Bước 7: Prediction với Probabilities

In [ ]:
# Dự đoán 1 mẫu mới
new_sample = np.array([[5.1, 3.5, 1.4, 0.2]])  # Iris-setosa
new_sample_scaled = scaler.transform(new_sample)
new_sample_tensor = torch.FloatTensor(new_sample_scaled)

with torch.no_grad():
    logits = model(new_sample_tensor)
    
    # Softmax để chuyển logits → probabilities
    probabilities = torch.softmax(logits, dim=1)
    
    # Lấy class có xác suất cao nhất
    predicted_class = torch.argmax(probabilities, dim=1)
    
    print(f"Logits: {logits[0]}")
    print(f"\nProbabilities:")
    for i, species in enumerate(label_encoder.classes_):
        print(f"  {species}: {probabilities[0][i]:.4f} ({probabilities[0][i]*100:.2f}%)")
    
    print(f"\n🎯 Dự đoán: {label_encoder.classes_[predicted_class]}")

---
## 📚 Key Concepts

### 1. Kiến trúc Logistic Regression
```python
Input (n features) → Linear Layer → Output (k classes)
                          ↓
                      y = W*x + b
```

### 2. Loss Function
- **CrossEntropyLoss**: Dùng cho multi-class classification
- Tự động tính Softmax → không cần thêm vào model
- Input: logits (raw scores), Target: class indices (0, 1, 2,...)

### 3. Softmax Function
```python
softmax(x_i) = exp(x_i) / sum(exp(x_j))
```
- Chuyển logits → probabilities (tổng = 1)
- Dùng khi muốn xem xác suất của từng class

### 4. Training Loop
```python
for epoch in range(num_epochs):
    outputs = model(X)           # Forward
    loss = criterion(outputs, y)  # Loss
    optimizer.zero_grad()         # Reset gradients
    loss.backward()               # Backward
    optimizer.step()              # Update
```

### 5. So sánh với Linear Regression
| | Linear Regression | Logistic Regression |
|---|---|---|
| Task | Regression | Classification |
| Output | Continuous | Discrete (classes) |
| Loss | MSE | CrossEntropy |
| Activation | None | Softmax |

### 6. Optimizer Options
```python
# SGD - Đơn giản, ổn định
optim.SGD(model.parameters(), lr=0.01)

# Adam - Thường tốt hơn, adaptive learning rate  
optim.Adam(model.parameters(), lr=0.01)

# SGD with momentum
optim.SGD(model.parameters(), lr=0.01, momentum=0.9)
```
